# 01 · Generative Agents: build a tiny memory loop

**Paper:** Park et al., *Generative Agents: Interactive Simulacra of Human Behavior* (2023).
[Paper, Sections 4.1–4.3](https://arxiv.org/html/2304.03442v2) ·
[Authors' implementation](https://github.com/joonspk-research/generative_agents)

This is an early foundational example of memory for LLM agents, not a claim that memory started in 2023.

**Prerequisites:** Python lists, dictionaries, functions, and sorting. No agent framework knowledge needed.
**Time:** roughly 60–90 minutes. **Runtime:** Python 3.10+, standard library only; no network, API key, model download, or GPU.

### What you will build

An agent remembers two observations about Maya, retrieves them, stores a general preference with evidence, and uses that preference when choosing a drink.

`observation → memory stream → retrieval → reflection → retrieval → action/plan → memory stream`

**Single learning claim:** your code can carry evidence through this entire memory loop.
**Main uncertainty:** can you implement the state changes, rather than just calculate a similarity score?
**Decisive check:** one deterministic, three-observation scenario at the end.
**Pass rule:** all exercise checks pass; the reflection cites the two tea observations; an empty-memory agent asks Maya, while the memory-equipped agent serves tea and stores its plan.
These are software checks on a constructed example, not evidence of an LLM quality improvement.

### How to work

1. Run the setup cell.
2. Read each function's contract, replace its `raise NotImplementedError(...)`, then run the check immediately below.
3. A `NotImplementedError` is an intentional exercise stop. A failed assertion means the implementation needs work.
4. Finish the final scenario. Use **Restart Kernel → Run All** to check that hidden notebook state is not helping.
5. Reference answers are collapsed at the end. They are Markdown, so **Run All never fills your answers for you**.

No cells write files. Keep this exercise source here; save executed copies, logs, and other run artifacts under
`/scratch/mingyan_root/mingyan/jmtang/outputs/dynamic_mem/tutorial/`.

### Paper mechanism versus teaching substitution

| Component | Paper | This lesson |
| --- | --- | --- |
| Memory | Text, creation/access times; observations, reflections, plans | Append-only list of dictionaries |
| Retrieval | Normalized recency + importance + semantic relevance | Same sum; word-count cosine replaces learned embeddings |
| Importance | Model rating from 1 to 10 | Supplied ratings |
| Recency | Hourly decay, factor 0.995 | Same formula; explicit numeric hours |
| Reflection | Importance-triggered questions, retrieved evidence, cited insights | One supplied question and a deterministic callback |
| Planning | Model-generated plans and reactions | One deterministic action, stored as a plan |

Paper reflection threshold: 150. Teaching threshold: 10, so three events suffice.
We use top-k instead of token-budget packing. Constant score columns normalize to zero; ties retain insertion order.
The scripted callbacks teach data flow; they do not reproduce model reasoning.
**Not now:** the 25-agent town, real LLM calls, recursive day planning, benchmark scores, training, or additional experiments.


In [9]:
import math
import re

VOCABULARY = ("maya", "tea", "coffee", "kitchen", "party")

def embed(text):
    """Teaching substitute: five word counts, not semantic embeddings."""
    words = re.findall(r"[a-z]+", text.lower())
    return [words.count(word) for word in VOCABULARY]

def assert_close(actual, expected):
    assert math.isclose(actual, expected, rel_tol=1e-9, abs_tol=1e-9), (actual, expected)

assert embed("Maya chooses tea, tea.") == [1, 2, 0, 0, 0]
print("Setup ready. Complete exercise 1 next.")


Setup ready. Complete exercise 1 next.


## 1 · Write one memory

A memory stream is ordinary external data; it does not update model weights.

Implement `append_memory`. Assign `id = len(stream)`, store all arguments, set both timestamps to `now`, convert evidence IDs to a tuple, append, and return the new dictionary. Preserve earlier records. Accept importance scores from 1 through 10; otherwise raise `ValueError`.

Times are simulated hours. IDs remain stable because this lesson never deletes records.
`created_at` answers “when did it happen?”; `last_accessed` answers “when did the agent last recall it?”

<details><summary>Hint</summary>

Build one dictionary, then call <code>stream.append(memory)</code>. A reflection uses the same structure as an observation.

</details>


In [ ]:
def append_memory(stream, text, now, importance, kind="observation", evidence=()):
    """Append a record and return it; importance must be in [1, 10]."""
    raise NotImplementedError("append_memory")


In [ ]:
write_stream = []
first = append_memory(write_stream, "Maya asks for tea.", 0, 8)
second = append_memory(write_stream, "Maya likes tea.", 1, 8, "reflection", [0])
assert first == {
    "id": 0, "text": "Maya asks for tea.", "created_at": 0,
    "last_accessed": 0, "importance": 8, "kind": "observation", "evidence": (),
}
assert second["id"] == 1 and second["evidence"] == (0,)
assert second["kind"] == "reflection" and write_stream[0] == first
for invalid_importance in (0, 11):
    try:
        append_memory(write_stream, "Invalid.", 2, invalid_importance)
    except ValueError:
        pass
    else:
        raise AssertionError("Out-of-range importance must be rejected")
assert len(write_stream) == 2
print("Exercise 1 passed.")


Exercise 1 passed.


## 2 · Measure relevance

Implement cosine similarity for two equal-length vectors:

`sum(left_i * right_i) / (length(left) * length(right))`

Here `length(vector) = sqrt(sum(value ** 2))`. Return 0.0 if either vector has zero length. Raise `ValueError` when dimensions differ. A zero vector is normal: our tiny vocabulary cannot represent every sentence.

<details><summary>Hint</summary>

Use <code>zip(left, right)</code> for the dot product and <code>math.sqrt</code> for each norm. Do not divide before checking zero norms.

</details>


In [ ]:
def cosine(left, right):
    """Return cosine similarity, with zero for a zero-norm vector."""
    raise NotImplementedError("cosine")


In [35]:
assert_close(cosine([1, 0], [1, 0]), 1.0)
assert_close(cosine([1, 0], [0, 1]), 0.0)
assert_close(cosine([1, 0], [-1, 0]), -1.0)
assert_close(cosine([3, 4], [3, 0]), 0.6)
assert cosine([0, 0], [1, 1]) == 0.0
assert cosine([], []) == 0.0
try:
    cosine([1], [1, 2])
except ValueError:
    pass
else:
    raise AssertionError("Different dimensions must be rejected")
assert_close(cosine(embed("Maya tea"), embed("Maya coffee")), 0.5)
print("Exercise 2 passed.")


Exercise 2 passed.


## 3 · Put scores on the same scale

Implement `(value - minimum) / (maximum - minimum)` for every value.
Return `[]` for empty input. Return all zeros when every value is equal: that feature cannot distinguish these candidates.
Do not modify the input list.

<details><summary>Hint</summary>

Handle empty and constant lists before division. Normalization happens across memories for each feature separately.

</details>


In [ ]:
def minmax(values):
    """Normalize a score column to [0, 1]; constant columns become zero."""
    raise NotImplementedError("minmax")


In [ ]:
scale_values = [2, 4, 6]
assert minmax(scale_values) == [0.0, 0.5, 1.0]
assert scale_values == [2, 4, 6]
assert minmax([-4, -2, 0]) == [0.0, 0.5, 1.0]
assert minmax([8, 8]) == [0.0, 0.0]
assert minmax([8]) == [0.0]
assert minmax([]) == []
print("Exercise 3 passed.")


## 4 · Combine three reasons to remember

Return one score per memory, in stream order:

- Raw recency: `0.995 ** (now - memory["last_accessed"])`.
- Raw importance: the stored rating.
- Raw relevance: `cosine(embed(query), embed(memory["text"]))`.

Apply `minmax` to each entire column, then add the three normalized values with weights 1, 1, 1.
Return `[]` for an empty stream. Reject `now < last_accessed` with `ValueError`.
Scoring must not change memory state.

<details><summary>Hint</summary>

Construct three lists, normalize them independently, then use <code>zip</code> to add the corresponding entries. Recency uses access time, not creation time.

</details>


In [ ]:
def memory_scores(stream, query, now):
    """Score every memory without changing timestamps or stream order."""
    raise NotImplementedError("memory_scores")


In [ ]:
score_stream = []
append_memory(score_stream, "tea", 0, 10)
append_memory(score_stream, "kitchen", 1, 1)
assert memory_scores(score_stream, "tea", 2) == [2.0, 1.0]
assert [memory["last_accessed"] for memory in score_stream] == [0, 1]
score_stream[0]["last_accessed"] = 2
assert memory_scores(score_stream, "tea", 2) == [3.0, 0.0]
assert memory_scores(score_stream, "kitchen", 2) == [2.0, 1.0]
assert memory_scores([], "tea", 2) == []
assert memory_scores(score_stream[:1], "tea", 2) == [0.0]
decay_stream = []
for timestamp in (0, 1, 2):
    append_memory(decay_stream, "tea", timestamp, 5)
decay_scores = memory_scores(decay_stream, "tea", 2)
assert_close(decay_scores[0], 0.0)
assert_close(decay_scores[1], 0.498746867167918)
assert_close(decay_scores[2], 1.0)
try:
    memory_scores(score_stream, "tea", 1)
except ValueError:
    pass
else:
    raise AssertionError("Time cannot precede last access")
print("Exercise 4 passed.")


## 5 · Retrieve, then update access time

Return at most `top_k` original memory dictionaries, ordered by descending score.
Break equal scores by insertion order. Score first, select second, then set **only selected memories'**
`last_accessed` to `now`. Preserve stream order and creation timestamps.

Return `[]` for empty memory or `top_k=0`; reject negative `top_k` with `ValueError`.
Assume `top_k` is an integer. Reflections and plans must be eligible for retrieval too.

<details><summary>Hint</summary>

Sort indices with <code>sorted(range(len(stream)), key=..., reverse=True)</code>. Python sorting is stable. Slice the indices, then update the selected dictionaries.

</details>


In [ ]:
def retrieve(stream, query, now, top_k=2):
    """Rank memories and refresh only the selected records."""
    raise NotImplementedError("retrieve")


In [ ]:
read_stream = []
append_memory(read_stream, "tea", 0, 10)
append_memory(read_stream, "kitchen", 1, 1)
selected = retrieve(read_stream, "tea", 2, top_k=1)
assert [memory["id"] for memory in selected] == [0]
assert selected[0] is read_stream[0]
assert [memory["last_accessed"] for memory in read_stream] == [2, 1]
assert [memory["created_at"] for memory in read_stream] == [0, 1]
assert [memory["id"] for memory in read_stream] == [0, 1]
assert retrieve(read_stream, "tea", 3, top_k=0) == []
assert [memory["last_accessed"] for memory in read_stream] == [2, 1]
assert retrieve([], "tea", 3) == []
tie_stream = []
append_memory(tie_stream, "tea", 0, 5)
append_memory(tie_stream, "tea", 0, 5, kind="reflection")
assert [memory["id"] for memory in retrieve(tie_stream, "tea", 1, 5)] == [0, 1]
try:
    retrieve(read_stream, "tea", 3, -1)
except ValueError:
    pass
else:
    raise AssertionError("Negative top_k must be rejected")
print("Exercise 5 passed.")


## Provided: two deliberately scripted model substitutes

Run this cell as supplied. These functions stand in for generation calls so you can inspect the memory plumbing.
They are hand-authored teaching rules, **not recorded LLM outputs**.

The reflector accepts a question and retrieved evidence and returns an insight with citations.
Here a **citation** is an evidence ID: the `id` of a memory the insight was derived from, not a literature reference.
The planner reads retrieved memory text. Its decision is deliberately simple and must not be interpreted as a measured benefit from reflection.
Neither function has access to the full stream or a hidden answer database.


In [ ]:
PREFERENCE = "Maya prefers tea over coffee."

def scripted_reflector(question, evidence):
    """Produce one toy insight only when the supplied evidence supports it."""
    tea_memories = [
        memory for memory in evidence
        if memory["kind"] == "observation"
        and {"maya", "tea"}.issubset(set(re.findall(r"[a-z]+", memory["text"].lower())))
    ]
    if len(tea_memories) < 2:
        raise ValueError("This teaching rule needs two observations about Maya and tea")
    return {
        "text": PREFERENCE,
        "importance": 8,
        "evidence": [memory["id"] for memory in tea_memories],
    }

def scripted_planner(situation, memories):
    """Choose a drink using only the supplied memory context."""
    if any(memory["text"] == PREFERENCE for memory in memories):
        return "Serve tea to Maya."
    return "Ask Maya what she would like."


## 6 · Turn observations into a reusable insight

Complete `reflect`, which returns `(new_reflection_or_None, remaining_importance)`.

`pending_importance` is a counter: the total importance of observations added since the previous reflection.
The caller adds each new observation's importance to it, passes it in, and replaces it with the returned `remaining_importance`.
It is unrelated to the `importance` of the reflection itself.

1. If `pending_importance <= threshold` or the stream is empty, return `(None, pending_importance)` without calling the generator.
2. Otherwise retrieve two memories for `question`, then call `generate(question, evidence)`.
3. The result is a dictionary with `text`, `importance`, and `evidence` (cited memory IDs). Require at least one citation and require every cited ID to be in the retrieved evidence; otherwise raise `ValueError`.
4. Store the insight with `append_memory`, using `kind="reflection"` and those citations, so it becomes a complete memory record. Return that record with counter 0.

Only a successful reflection resets the counter. Citation validation checks provenance, not whether the generated claim is true.

<details><summary>Hint</summary>

Collect retrieved IDs in a set. Use <code>set(insight["evidence"]).issubset(available_ids)</code> before appending. The callback supplies the insight; your task is the memory lifecycle.

</details>


In [ ]:
def reflect(stream, pending_importance, now, question, generate, threshold=10):
    """Store a cited insight when accumulated observation importance exceeds the threshold."""
    raise NotImplementedError("reflect")


In [ ]:
reflection_stream = []
append_memory(reflection_stream, "Maya asks for tea.", 0, 8)
append_memory(reflection_stream, "Maya chooses tea again.", 1, 8)

def should_not_generate(question, evidence):
    raise AssertionError("Reflection must not run at or below the threshold")

assert reflect(reflection_stream, 10, 2, "Maya tea", should_not_generate) == (None, 10)
assert [memory["last_accessed"] for memory in reflection_stream] == [0, 1]
assert reflect([], 11, 2, "Maya tea", should_not_generate) == (None, 11)
insight, remaining = reflect(reflection_stream, 16, 2, "Maya tea", scripted_reflector)
assert insight["kind"] == "reflection"
assert insight["text"] == PREFERENCE
assert set(insight["evidence"]) == {0, 1}
assert remaining == 0 and len(reflection_stream) == 3
assert insight["created_at"] == insight["last_accessed"] == 2
assert reflect(reflection_stream, remaining, 3, "Maya tea", should_not_generate) == (None, 0)
assert retrieve(reflection_stream, "Maya tea coffee", 3, 1)[0] is insight

for invalid_evidence in ([], [999]):
    def invalid_reflector(question, evidence):
        return {"text": "Unsupported.", "importance": 8, "evidence": invalid_evidence}

    try:
        reflect(reflection_stream, 11, 4, "Maya tea", invalid_reflector)
    except ValueError:
        pass
    else:
        raise AssertionError("Missing or unavailable citations must be rejected")
assert len(reflection_stream) == 3
print("Exercise 6 passed.")


## 7 · Let memory influence an action

Implement one small planning step:

1. Retrieve two memories for the current `situation`.
2. Call `plan(situation, memories)` to obtain an action string.
3. Store the action as a memory of kind `"plan"`, with teaching importance 5.
4. Return the action string.

This deliberately reduces planning to a single action. The external environment would execute it and supply a new observation; recording a plan alone does not prove the action happened.

<details><summary>Hint</summary>

This function needs only three calls: <code>retrieve</code>, <code>plan</code>, and <code>append_memory</code>. Do not hard-code an action into this function.

</details>


In [ ]:
def agent_step(stream, situation, now, plan):
    """Retrieve context, choose an action, and remember the plan."""
    raise NotImplementedError("agent_step")


In [ ]:
empty_agent_stream = []
assert agent_step(empty_agent_stream, "Maya tea coffee", 0, scripted_planner) == "Ask Maya what she would like."
assert empty_agent_stream[-1]["kind"] == "plan"
assert empty_agent_stream[-1]["importance"] == 5
acting_stream = []
append_memory(acting_stream, PREFERENCE, 0, 8, kind="reflection")
assert agent_step(acting_stream, "Maya tea coffee", 1, scripted_planner) == "Serve tea to Maya."
assert acting_stream[0]["last_accessed"] == 1
assert acting_stream[-1]["text"] == "Serve tea to Maya."

def another_planner(situation, memories):
    assert situation == "party"
    assert [memory["text"] for memory in memories] == ["party"]
    return "Send an invitation."

other_stream = []
append_memory(other_stream, "party", 0, 8)
assert agent_step(other_stream, "party", 1, another_planner) == "Send an invitation."
assert other_stream[-1]["text"] == "Send an invitation."
print("Exercise 7 passed.")


## Final check · One complete memory episode

The scenario below is the only end-to-end demonstration. Predict the two actions before running it.

The three observations and ratings are hand-written teaching inputs. We test the trigger after each observation.
The second observation crosses the teaching threshold; a reflection is written and the counter resets.
The final minor observation remains pending and must not cause another reflection.

For the acceptance check, we record the counter immediately after the successful reflection.
We then compare an empty-memory branch with the populated-memory branch using the same situation and planner.


In [ ]:
demo_stream = []
pending_importance = 0
events = [
    ("Maya asks for tea.", 0, 8),
    ("Maya chooses tea again.", 1, 8),
    ("The kitchen floor is clean.", 2, 1),
]
counter_after_reflection = None
for text, now, importance in events:
    append_memory(demo_stream, text, now, importance)
    pending_importance += importance
    reflection, pending_importance = reflect(
        demo_stream, pending_importance, now, "Maya tea", scripted_reflector,
    )
    if reflection is not None:
        counter_after_reflection = pending_importance

assert counter_after_reflection == 0
assert pending_importance == 1
reflections = [memory for memory in demo_stream if memory["kind"] == "reflection"]
assert len(reflections) == 1
assert set(reflections[0]["evidence"]) == {0, 1}

situation = "Maya tea coffee"
without_memory = agent_step([], situation, 3, scripted_planner)
with_memory = agent_step(demo_stream, situation, 3, scripted_planner)

assert without_memory == "Ask Maya what she would like."
assert with_memory == "Serve tea to Maya."
assert demo_stream[-1]["kind"] == "plan"
assert demo_stream[-1]["text"] == with_memory

print("Without memory:", without_memory)
print("With memory:   ", with_memory)
print("Memory stream:")
for memory in demo_stream:
    print(memory["id"], memory["kind"], memory["text"], "evidence:", memory["evidence"])
print("The teaching memory loop passed. This is not a paper benchmark.")


## Explain it back to yourself

Answer in the Markdown cell below. No extra runs are needed.

1. Why can an old memory have high recency after retrieval?
2. Why is a reflection stored in the same stream as observations?
3. The demo's reflection cites memories 0 and 1. What does this citation prove, and what does it fail to prove?
4. Which functions would you replace to use a real embedding model and LLM?
5. Why does this scripted action difference not establish the paper's claims?
6. After serving tea, where would you write the environment's observation?

Completion means you can trace an observation into a cited reflection, then into retrieved context and a stored plan.
A database alone is storage; this exercise adds the decisions about what to recall and how to reuse it.


### Your explanations

1. …
2. …
3. …
4. …
5. …
6. …


## Reference answers · Open only after trying

Copy one answer into its corresponding exercise cell if needed, then restart and rerun.
These are plain Markdown blocks and never execute automatically.


<details>
<summary>Reference answer: append_memory</summary>

```python
def append_memory(stream, text, now, importance, kind="observation", evidence=()):
    if not 1 <= importance <= 10:
        raise ValueError("importance must be in [1, 10]")
    memory = {
        "id": len(stream),
        "text": text,
        "created_at": now,
        "last_accessed": now,
        "importance": importance,
        "kind": kind,
        "evidence": tuple(evidence),
    }
    stream.append(memory)
    return memory
```

</details>


<details>
<summary>Reference answer: cosine</summary>

```python
def cosine(left, right):
    if len(left) != len(right):
        raise ValueError("vectors must have equal dimensions")
    left_norm = math.sqrt(sum(value ** 2 for value in left))
    right_norm = math.sqrt(sum(value ** 2 for value in right))
    if left_norm == 0 or right_norm == 0:
        return 0.0
    dot_product = sum(left_value * right_value for left_value, right_value in zip(left, right))
    return dot_product / (left_norm * right_norm)
```

</details>


<details>
<summary>Reference answer: minmax</summary>

```python
def minmax(values):
    if not values:
        return []
    minimum = min(values)
    maximum = max(values)
    if maximum == minimum:
        return [0.0] * len(values)
    return [(value - minimum) / (maximum - minimum) for value in values]
```

</details>


<details>
<summary>Reference answer: memory_scores</summary>

```python
def memory_scores(stream, query, now):
    if any(now < memory["last_accessed"] for memory in stream):
        raise ValueError("now must not precede last access")
    recency = minmax([0.995 ** (now - memory["last_accessed"]) for memory in stream])
    importance = minmax([memory["importance"] for memory in stream])
    relevance = minmax([cosine(embed(query), embed(memory["text"])) for memory in stream])
    return [
        recent + important + relevant
        for recent, important, relevant in zip(recency, importance, relevance)
    ]
```

</details>


<details>
<summary>Reference answer: retrieve</summary>

```python
def retrieve(stream, query, now, top_k=2):
    if top_k < 0:
        raise ValueError("top_k must be nonnegative")
    if not stream or top_k == 0:
        return []
    scores = memory_scores(stream, query, now)
    ranked_indices = sorted(range(len(stream)), key=lambda index: scores[index], reverse=True)
    selected = [stream[index] for index in ranked_indices[:top_k]]
    for memory in selected:
        memory["last_accessed"] = now
    return selected
```

</details>


<details>
<summary>Reference answer: reflect</summary>

```python
def reflect(stream, pending_importance, now, question, generate, threshold=10):
    if pending_importance <= threshold or not stream:
        return None, pending_importance
    evidence = retrieve(stream, question, now, top_k=2)
    insight = generate(question, evidence)
    available_ids = {memory["id"] for memory in evidence}
    if not insight["evidence"] or not set(insight["evidence"]).issubset(available_ids):
        raise ValueError("reflection must cite retrieved memories")
    memory = append_memory(
        stream, insight["text"], now, insight["importance"],
        kind="reflection", evidence=insight["evidence"],
    )
    return memory, 0
```

</details>


<details>
<summary>Reference answer: agent_step</summary>

```python
def agent_step(stream, situation, now, plan):
    memories = retrieve(stream, situation, now, top_k=2)
    action = plan(situation, memories)
    append_memory(stream, action, now, importance=5, kind="plan")
    return action
```

</details>
